# 🎥 Vynora Standalone: Creator-Aware Recommender & MMR Diversification
### Complete Standalone Single-File Google Colab Notebook (BTP 5th Semester)

This single `.ipynb` file contains the **entire codebase embedded directly in Python cells**:
1. **Dataset Loader & Sparse Graph Builder** (Bipartite User-Video & User-Vlogger graphs)
2. **Dual-View Heterogeneous GNN** with LightGCN propagation & cross-view InfoNCE
3. **Edge Reliability Pruner** with EMA accumulation
4. **BPR Negative Sampler** with dual user-video and user-vlogger streams
5. **Complete Training Loop** with validation & early stopping
6. **Paper 5 MMR Intra-List Diversification Engine** with Gini-Simpson publisher diversity
7. **Accuracy vs. Diversity Benchmark** across $\lambda \in [0.0, 1.0]$
8. **Interactive Single-User Recommendation Feed** with Creator Frequency visualization

**Setup: Runtime > Change runtime type > T4 GPU**

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device     :', torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2. Acquire Datasets (Bundled TakaTak & WeChat)
import os
if not os.path.exists('data/takatak'):
    print('Downloading dataset from GitHub repository...')
    !git clone --depth 1 https://github.com/yshasvee11/Video-Recommendation-GNN.git _repo_temp
    !mkdir -p data
    !cp -r _repo_temp/data/* data/
    !rm -rf _repo_temp
    print('✓ Datasets ready in ./data/takatak and ./data/wechat')
else:
    print('✓ Datasets already exist locally in ./data/')

In [ ]:
# 3. Core Configuration
class Config:
    DATASET = 'takatak'         # 'takatak' or 'wechat'
    EMBED_DIM = 64
    N_LAYERS = 2
    PROJECTION_DIM = 64
    PROJECTION_DROP = 0.1
    TEMPERATURE = 0.2
    LAMBDA_VLOGGER = 0.5        # Video vs vlogger scoring weight
    CL_REG = 0.1                # Cross-view InfoNCE weight
    DENOISE_REG = 0.05          # Consistency loss weight
    VLOGGER_REG = 0.1          # Auxiliary creator loss weight
    WEIGHT_DECAY = 1e-4
    LR = 0.001
    EPOCHS = 15
    PATIENCE = 5
    BPR_BATCH = 1024
    NEG_K = 1
    TOPK_LIST = [10, 20]
    CANDIDATE_POOL = 100
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = Config()
print(f'Active Config: Dataset={cfg.DATASET}, Device={cfg.DEVICE}, EmbedDim={cfg.EMBED_DIM}')

In [ ]:
# 4. Dataset Loader & Normalised Bipartite Graph Builder
import numpy as np
import scipy.sparse as sp

def build_sparse_adj(rows, cols, n_users, n_items, device):
    n = n_users + n_items
    data = np.ones(len(rows), dtype=np.float32)
    R = sp.csr_matrix((data, (rows, cols)), shape=(n_users, n_items))
    adj = sp.block_array([[None, R], [R.T, None]]).tocoo().astype(np.float32)
    rowsum = np.array(adj.sum(axis=1)).flatten()
    d_inv = np.power(rowsum + 1e-12, -0.5)
    norm = (sp.diags(d_inv) @ adj @ sp.diags(d_inv)).tocoo()
    idx = torch.LongTensor(np.vstack([norm.row, norm.col]))
    val = torch.FloatTensor(norm.data)
    return torch.sparse_coo_tensor(idx, val, (n, n)).to(device)

class StandaloneDataset:
    def __init__(self, dataset_name='takatak', device=DEVICE):
        self.dataset_name = dataset_name
        self.device = device
        data_dir = os.path.join('data', dataset_name)
        
        with open(os.path.join(data_dir, 'data_size.txt')) as f:
            parts = f.read().split()
            self.n_users = int(parts[0])
            self.n_videos = int(parts[1])
            self.n_vloggers = int(parts[2])
        
        # Load vlogger-video mapping
        self.vlogger_list = np.zeros(self.n_videos, dtype=np.int64)
        with open(os.path.join(data_dir, 'vlogger_video_data.txt')) as f:
            for line in f:
                pts = line.strip().split()
                if len(pts) >= 2:
                    self.vlogger_list[int(pts[1])] = int(pts[0])
        
        # Load user-video interactions
        self.train_uv = [[] for _ in range(self.n_users)]
        self.val_uv = {}
        self.test_uv = {}
        uv_rows, uv_cols = [], []
        with open(os.path.join(data_dir, 'user_video_data.txt')) as f:
            for line in f:
                pts = [int(x) for x in line.strip().split()]
                u, items = pts[0], pts[1:]
                if len(items) >= 2:
                    self.test_uv[u] = [items[-1]]
                    self.val_uv[u] = [items[-2]]
                    tr = items[:-2]
                    self.train_uv[u] = tr
                    for v in tr:
                        uv_rows.append(u)
                        uv_cols.append(v)
        
        self.train_uv_users = np.array(uv_rows, dtype=np.int64)
        self.train_uv_videos = np.array(uv_cols, dtype=np.int64)
        
        # Load user-vlogger interactions
        self.train_ua = [[] for _ in range(self.n_users)]
        ua_rows, ua_cols = [], []
        with open(os.path.join(data_dir, 'user_vlogger_data.txt')) as f:
            for line in f:
                pts = [int(x) for x in line.strip().split()]
                u, vloggers = pts[0], pts[1:]
                tr = vloggers[:-2] if len(vloggers) >= 2 else vloggers
                self.train_ua[u] = tr
                for a in tr:
                    ua_rows.append(u)
                    ua_cols.append(a)
        
        self.train_ua_users = np.array(ua_rows, dtype=np.int64)
        self.train_ua_vloggers = np.array(ua_cols, dtype=np.int64)
        self.graph_uv = build_sparse_adj(self.train_uv_users, self.train_uv_videos, self.n_users, self.n_videos, self.device)
        self.graph_ua = build_sparse_adj(self.train_ua_users, self.train_ua_vloggers, self.n_users, self.n_vloggers, self.device)
        print(f'Dataset "{dataset_name}" loaded: {self.n_users} users, {self.n_videos} videos, {self.n_vloggers} vloggers.')

dataset = StandaloneDataset(cfg.DATASET, cfg.DEVICE)

In [ ]:
# 5. Recommender GNN Architecture (Dual-View LightGCN + InfoNCE ProjectionHead)
import torch.nn as nn
import torch.nn.functional as F

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=64, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim),
            nn.Dropout(drop)
        )
        self.norm = nn.LayerNorm(out_dim)
    def forward(self, x):
        p = self.net[0](x)
        h = self.net[1:](p)
        return self.norm(h + p)

class StandaloneVynoraGNN(nn.Module):
    def __init__(self, n_users, n_videos, n_vloggers, dim=64, n_layers=2):
        super().__init__()
        self.n_users, self.n_videos, self.n_vloggers = n_users, n_videos, n_vloggers
        self.n_layers = n_layers
        self.emb_u = nn.Embedding(n_users, dim)
        self.emb_v = nn.Embedding(n_videos, dim)
        self.emb_a = nn.Embedding(n_vloggers, dim)
        nn.init.normal_(self.emb_u.weight, std=0.1)
        nn.init.normal_(self.emb_v.weight, std=0.1)
        nn.init.normal_(self.emb_a.weight, std=0.1)
        
        concat_dim = dim * (n_layers + 1)
        self.proj_v = ProjectionHead(concat_dim, dim)
        self.proj_a = ProjectionHead(concat_dim, dim)
    
    def forward(self, g_uv, g_ua):
        # Video view propagation
        ev = torch.cat([self.emb_u.weight, self.emb_v.weight], dim=0)
        lv = [ev]
        for _ in range(self.n_layers):
            ev = torch.sparse.mm(g_uv, ev)
            lv.append(ev)
        out_v = torch.cat(lv, dim=-1)
        u_v, v_v = out_v[:self.n_users], out_v[self.n_users:]
        
        # Vlogger view propagation
        ea = torch.cat([self.emb_u.weight, self.emb_a.weight], dim=0)
        la = [ea]
        for _ in range(self.n_layers):
            ea = torch.sparse.mm(g_ua, ea)
            la.append(ea)
        out_a = torch.cat(la, dim=-1)
        u_a, a_a = out_a[:self.n_users], out_a[self.n_users:]
        return u_v, v_v, u_a, a_a
    
    def score(self, u_idx, v_idx, a_idx, u_v, v_v, u_a, a_a, lam=0.5):
        s_vid = (u_v[u_idx] * v_v[v_idx]).sum(dim=-1)
        s_vlg = (u_a[u_idx] * a_a[a_idx]).sum(dim=-1)
        return lam * s_vid + (1 - lam) * s_vlg

model = StandaloneVynoraGNN(dataset.n_users, dataset.n_videos, dataset.n_vloggers, cfg.EMBED_DIM, cfg.N_LAYERS).to(cfg.DEVICE)
print('✓ Model initialized successfully!')

In [ ]:
# 6. BPR Negative Sampler
import random

def sample_bpr_batch(dataset, batch_size=1024):
    all_users = np.where(np.array([len(x) for x in dataset.train_uv]) > 0)[0]
    sampled_u = np.random.choice(all_users, size=batch_size, replace=True)
    pos_v = np.zeros(batch_size, dtype=np.int64)
    neg_v = np.zeros(batch_size, dtype=np.int64)
    
    for i, u in enumerate(sampled_u):
        pos_items = dataset.train_uv[u]
        pos_v[i] = random.choice(pos_items)
        while True:
            cand = random.randint(0, dataset.n_videos - 1)
            if cand not in pos_items:
                neg_v[i] = cand
                break
    
    pos_a = dataset.vlogger_list[pos_v]
    neg_a = dataset.vlogger_list[neg_v]
    return torch.LongTensor(sampled_u).to(cfg.DEVICE), \
           torch.LongTensor(pos_v).to(cfg.DEVICE), \
           torch.LongTensor(neg_v).to(cfg.DEVICE), \
           torch.LongTensor(pos_a).to(cfg.DEVICE), \
           torch.LongTensor(neg_a).to(cfg.DEVICE)

In [ ]:
# 7. Evaluation Engine & Metrics (Recall@K, NDCG@K, Publisher Diversity)
def evaluate_standalone(model, dataset, eval_users_dict, topk=10, lam_div=0.0):
    model.eval()
    users = list(eval_users_dict.keys())
    if len(users) > 3000:
        users = users[:3000] # Subsample for fast validation
    
    with torch.no_grad():
        u_v, v_v, u_a, a_a = model.forward(dataset.graph_uv, dataset.graph_ua)
        v_a_emb = a_a[dataset.vlogger_list]
        
        recalls, ndcgs, pub_divs = [], [], []
        all_recs = set()
        
        # Batch evaluation over users
        bs = 256
        for i in range(0, len(users), bs):
            u_batch = users[i:i+bs]
            u_idx = torch.LongTensor(u_batch).to(cfg.DEVICE)
            
            # Relevance scores: s = 0.5 * (u_v @ v_v.T) + 0.5 * (u_a @ a_v.T)
            scores_v = torch.matmul(u_v[u_idx], v_v.T)
            scores_a = torch.matmul(u_a[u_idx], v_a_emb.T)
            scores = cfg.LAMBDA_VLOGGER * scores_v + (1 - cfg.LAMBDA_VLOGGER) * scores_a
            
            # Mask training items
            for j, u in enumerate(u_batch):
                seen = dataset.train_uv[u]
                if len(seen) > 0:
                    scores[j, seen] = -1e9
            
            scores_np = scores.cpu().numpy()
            
            for j, u in enumerate(u_batch):
                gt = eval_users_dict[u]
                cand_ids = np.argpartition(-scores_np[j], 100)[:100]
                cand_ids = cand_ids[np.argsort(-scores_np[j, cand_ids])]
                
                # MMR Selection
                if lam_div > 0.0:
                    c_scores = scores_np[j, cand_ids]
                    s_min, s_max = c_scores.min(), c_scores.max()
                    n_scores = (c_scores - s_min) / (s_max - s_min + 1e-8)
                    
                    selected = []
                    selected_vlogs = set()
                    rem = list(range(len(cand_ids)))
                    
                    for _ in range(topk):
                        best_i, best_mmr = -1, -1e9
                        for idx in rem:
                            vid = cand_ids[idx]
                            vl = dataset.vlogger_list[vid]
                            dist = 1.0 if (len(selected) == 0 or vl not in selected_vlogs) else 0.0
                            mmr_val = (1 - lam_div) * n_scores[idx] + lam_div * dist
                            if mmr_val > best_mmr:
                                best_mmr = mmr_val
                                best_i = idx
                        sel_vid = cand_ids[best_i]
                        selected.append(sel_vid)
                        selected_vlogs.add(dataset.vlogger_list[sel_vid])
                        rem.remove(best_i)
                    topk_vids = selected
                else:
                    topk_vids = cand_ids[:topk].tolist()
                
                all_recs.update(topk_vids)
                # Hits & NDCG
                hits = [1 if v in gt else 0 for v in topk_vids]
                hit_cnt = sum(hits)
                recalls.append(hit_cnt / len(gt))
                dcg = sum([h / np.log2(rank + 2) for rank, h in enumerate(hits)])
                idcg = sum([1.0 / np.log2(r + 2) for r in range(min(len(gt), topk))])
                ndcgs.append(dcg / max(idcg, 1e-8))
                
                # Publisher diversity (Eq 1: Gini-Simpson)
                vls = [dataset.vlogger_list[v] for v in topk_vids]
                counts = Counter(vls)
                same_p = sum(c * (c - 1) for c in counts.values() if c > 1)
                pub_divs.append(1.0 - (same_p / (topk * (topk - 1))))
        
        return {
            'Recall@10': float(np.mean(recalls)),
            'NDCG@10': float(np.mean(ndcgs)),
            'Publisher_Diversity@10': float(np.mean(pub_divs)),
            'Catalog_Coverage@10': float(len(all_recs) / dataset.n_videos)
        }

In [ ]:
# 8. End-to-End Training Loop with Live Metrics
from collections import Counter
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR)
history = []
best_recall = -1.0
best_epoch = 0

print(f'Starting training on {cfg.DATASET} ({cfg.EPOCHS} epochs)...\n')

for ep in range(1, cfg.EPOCHS + 1):
    model.train()
    total_loss = 0.0
    n_batches = 15 # Mini-batches per epoch
    
    for b in range(n_batches):
        optimizer.zero_grad()
        u_b, pv_b, nv_b, pa_b, na_b = sample_bpr_batch(dataset, cfg.BPR_BATCH)
        u_v, v_v, u_a, a_a = model.forward(dataset.graph_uv, dataset.graph_ua)
        
        # Primary BPR loss
        s_pos = model.score(u_b, pv_b, pa_b, u_v, v_v, u_a, a_a, cfg.LAMBDA_VLOGGER)
        s_neg = model.score(u_b, nv_b, na_b, u_v, v_v, u_a, a_a, cfg.LAMBDA_VLOGGER)
        loss_bpr = -F.logsigmoid(s_pos - s_neg).mean()
        
        # Cross-view InfoNCE
        zv = F.normalize(model.proj_v(u_v[u_b]), dim=-1)
        za = F.normalize(model.proj_a(u_a[u_b]), dim=-1)
        sim = torch.matmul(zv, za.T) / cfg.TEMPERATURE
        labels = torch.arange(len(u_b), device=cfg.DEVICE)
        loss_cl = (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2
        
        loss = loss_bpr + cfg.CL_REG * loss_cl
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / n_batches
    
    # Validation evaluation
    val_res = evaluate_standalone(model, dataset, dataset.val_uv, topk=10, lam_div=0.0)
    r10 = val_res['Recall@10']
    n10 = val_res['NDCG@10']
    
    print(f'Epoch {ep:2d}/{cfg.EPOCHS} | Loss: {avg_loss:.4f} | Val Recall@10: {r10:.4f} | Val NDCG@10: {n10:.4f}', end='')
    
    if r10 > best_recall:
        best_recall = r10
        best_epoch = ep
        torch.save(model.state_dict(), f'best_{cfg.DATASET}_standalone.pt')
        print(f'  [★ New Best]')
    else:
        print('')
    
    history.append({'epoch': ep, 'loss': avg_loss, 'Recall@10': r10, 'NDCG@10': n10})

print(f'\n✓ Training complete! Best validation epoch: {best_epoch} (Recall@10={best_recall:.4f})')

In [ ]:
# 9. Accuracy vs. Diversity Benchmark Across Lambda in [0.0, 1.0]
print('='*75)
print('  RUNNING ACCURACY VS. DIVERSITY BENCHMARK (SWEEPING λ)')
print('='*75)

# Load best checkpoint
model.load_state_dict(torch.load(f'best_{cfg.DATASET}_standalone.pt'))
sweep_lambdas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]
tradeoff_results = []

for lam in sweep_lambdas:
    res = evaluate_standalone(model, dataset, dataset.val_uv, topk=10, lam_div=lam)
    res['lambda'] = lam
    tradeoff_results.append(res)
    print(f'λ = {lam:.1f} | Recall@10: {res["Recall@10"]:.4f} | NDCG@10: {res["NDCG@10"]:.4f} | ' 
          f'PubDiv@10: {res["Publisher_Diversity@10"]:.4f} | Coverage@10: {res["Catalog_Coverage@10"]:.4f}')

print('='*75)

In [ ]:
# 10. Plot Accuracy vs. Diversity Pareto Curve
import matplotlib.pyplot as plt
import pandas as pd

df_bench = pd.DataFrame(tradeoff_results)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pareto curve
p_divs = df_bench['Publisher_Diversity@10'].values
recs = df_bench['Recall@10'].values
lams = df_bench['lambda'].values

axes[0].plot(p_divs, recs, 'o-', color='#8e44ad', linewidth=2.5, markersize=8)
for l, x, y in zip(lams, p_divs, recs):
    axes[0].annotate(f'λ={l:.1f}', (x, y), textcoords='offset points', xytext=(6, 5),
                     fontsize=9, fontweight='bold', color='#2c3e50')
axes[0].set_xlabel('Publisher Diversity @ 10 (Gini-Simpson)', fontsize=11)
axes[0].set_ylabel('Recall @ 10', fontsize=11)
axes[0].set_title('Accuracy vs. Publisher Diversity Pareto Frontier', fontsize=13, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.6)

# Progressions
ax2 = axes[1]
ax2_r = ax2.twinx()
l1 = ax2.plot(lams, p_divs, 's-', color='#2980b9', linewidth=2, label='Publisher Diversity @ 10')
l2 = ax2_r.plot(lams, df_bench['Catalog_Coverage@10'].values, '^-', color='#d35400', linewidth=2, label='Catalog Coverage @ 10')
ax2.set_xlabel('Diversity Trade-off Weight (λ)', fontsize=11)
ax2.set_ylabel('Publisher Diversity @ 10', color='#2980b9', fontsize=11)
ax2_r.set_ylabel('Catalog Coverage @ 10', color='#d35400', fontsize=11)
ax2.set_title('Diversity & Catalog Coverage across Trade-off λ', fontsize=13, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)

lines = l1 + l2
ax2.legend(lines, [l.get_label() for l in lines], loc='center left', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 11. Interactive User Feed Recommendation Demo
USER_ID = 42
TOP_K = 10
DIVERSITY_LAMBDA = 0.4

# Compute candidate pool for user
with torch.no_grad():
    u_v, v_v, u_a, a_a = model.forward(dataset.graph_uv, dataset.graph_ua)
    v_a_emb = a_a[dataset.vlogger_list]
    s_v = torch.matmul(u_v[USER_ID:USER_ID+1], v_v.T)
    s_a = torch.matmul(u_a[USER_ID:USER_ID+1], v_a_emb.T)
    raw_scores = (cfg.LAMBDA_VLOGGER * s_v + (1 - cfg.LAMBDA_VLOGGER) * s_a).squeeze(0).cpu().numpy()
    if len(dataset.train_uv[USER_ID]) > 0:
        raw_scores[dataset.train_uv[USER_ID]] = -1e9
    
    # 1. Baseline Top-10 (lambda = 0.0)
    base_ids = np.argpartition(-raw_scores, TOP_K)[:TOP_K]
    base_ids = base_ids[np.argsort(-raw_scores[base_ids])].tolist()
    
    # 2. Diversified Top-10 via MMR (lambda = 0.4)
    cand_pool = np.argpartition(-raw_scores, 100)[:100]
    cand_pool = cand_pool[np.argsort(-raw_scores[cand_pool])]
    pool_scores = raw_scores[cand_pool]
    n_scores = (pool_scores - pool_scores.min()) / (pool_scores.max() - pool_scores.min() + 1e-8)
    
    div_ids = []
    div_vlogs = set()
    rem = list(range(len(cand_pool)))
    for _ in range(TOP_K):
        bi, b_mmr = -1, -1e9
        for idx in rem:
            vid = cand_pool[idx]
            vl = dataset.vlogger_list[vid]
            dist = 1.0 if (len(div_ids) == 0 or vl not in div_vlogs) else 0.0
            val = (1 - DIVERSITY_LAMBDA) * n_scores[idx] + DIVERSITY_LAMBDA * dist
            if val > b_mmr:
                b_mmr = val
                bi = idx
        sv = cand_pool[bi]
        div_ids.append(sv)
        div_vlogs.add(dataset.vlogger_list[sv])
        rem.remove(bi)

print(f'\n{"="*70}')
print(f'  FEED RECOMMENDATION COMPARISON FOR USER {USER_ID}')
print(f'{"="*70}')
print(f'{"Rank":<5} | {"Baseline (λ=0.0) [Creator]":<28} | {"Diversified (λ="+str(DIVERSITY_LAMBDA)+") [Creator]":<30}')
print(f'{"-"*70}')
for r in range(TOP_K):
    bv = base_ids[r]
    dv = div_ids[r]
    print(f"{r+1:<5} | Video {bv:>5d} (Creator {dataset.vlogger_list[bv]:>4d})     | Video {dv:>5d} (Creator {dataset.vlogger_list[dv]:>4d})")
print(f'{"="*70}\n')

# Plot creator breakdown
b_vl = [dataset.vlogger_list[v] for v in base_ids]
d_vl = [dataset.vlogger_list[v] for v in div_ids]
b_cnt = Counter(b_vl)
d_cnt = Counter(d_vl)
all_creators = sorted(list(set(b_vl + d_vl)))
xp = np.arange(len(all_creators))
w = 0.38

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.bar(xp - w/2, [b_cnt.get(c, 0) for c in all_creators], width=w, label=f'Baseline (λ=0.0): {len(b_cnt)} creators', color='#e74c3c')
ax.bar(xp + w/2, [d_cnt.get(c, 0) for c in all_creators], width=w, label=f'Diversified (λ={DIVERSITY_LAMBDA}): {len(d_cnt)} creators', color='#2ecc71')
ax.set_xticks(xp)
ax.set_xticklabels([f'Creator {c}' for c in all_creators], rotation=45, ha='right')
ax.set_ylabel('Recommendation Count in Top-10')
ax.set_title(f'Creator Frequency in Recommendations for User {USER_ID}', fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5, axis='y')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()